# Module 9: Integrating Polars into the DataScience Workflow

In [1]:
import polars as pl

In [2]:
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

In [3]:
zone_columns_rename_mapping = {
    "LocationID": "location_id",
    "Borough": "borough",
    "Zone": "zone",
}

zones_df = (
    pl.read_parquet("./../data/taxi_zone_lookup.parquet")
).rename(zone_columns_rename_mapping)

zones_df.head()

location_id,borough,zone,service_zone
i32,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""


In [4]:
yellow_rides_column_rename_mapping = {
    "VendorID": "vendor_id",
    "RatecodeID": "rate_code_id",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "Airport_fee": "airport_fee",
}

zones_df_columns = ["borough", "zone", "service_zone"]
rides_df_raw = (
    pl.read_parquet("./../data/yellow_tripdata_2024-*.parquet")
    .rename(yellow_rides_column_rename_mapping)
    .join(zones_df, left_on="pickup_location_id", right_on="location_id")
    .rename({zones_df_column: f"pu_{zones_df_column}" for zones_df_column in zones_df_columns})
    .join(zones_df, left_on="dropoff_location_id", right_on="location_id")
    .rename({zones_df_column: f"do_{zones_df_column}" for zones_df_column in zones_df_columns})
)

rides_df_raw.head()

vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pu_borough,pu_zone,pu_service_zone,do_borough,do_zone,do_service_zone
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str
2,2024-02-01 00:04:45,2024-02-01 00:19:58,1,4.39,1,"""N""",68,236,1,20.5,1.0,0.5,1.28,0.0,1.0,26.78,2.5,0.0,"""Manhattan""","""East Chelsea""","""Yellow Zone""","""Manhattan""","""Upper East Side North""","""Yellow Zone"""
2,2024-02-01 00:56:31,2024-02-01 01:10:53,1,7.71,1,"""N""",48,243,1,31.0,1.0,0.5,9.0,0.0,1.0,45.0,2.5,0.0,"""Manhattan""","""Clinton East""","""Yellow Zone""","""Manhattan""","""Washington Heights North""","""Boro Zone"""
2,2024-02-01 00:07:50,2024-02-01 00:43:12,2,28.69,2,"""N""",132,261,2,70.0,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75,"""Queens""","""JFK Airport""","""Airports""","""Manhattan""","""World Trade Center""","""Yellow Zone"""
1,2024-02-01 00:01:49,2024-02-01 00:10:47,1,1.1,1,"""N""",161,163,1,9.3,3.5,0.5,2.85,0.0,1.0,17.15,2.5,0.0,"""Manhattan""","""Midtown Center""","""Yellow Zone""","""Manhattan""","""Midtown North""","""Yellow Zone"""
1,2024-02-01 00:37:35,2024-02-01 00:51:15,1,2.6,1,"""N""",246,79,2,15.6,3.5,0.5,0.0,0.0,1.0,20.6,2.5,0.0,"""Manhattan""","""West Chelsea/Hudson Yards""","""Yellow Zone""","""Manhattan""","""East Village""","""Yellow Zone"""


In [5]:
rides_df_raw.shape

(6590154, 25)

## Data Exploration

In [6]:
rides_df_raw.describe()

statistic,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pu_borough,pu_zone,pu_service_zone,do_borough,do_zone,do_service_zone
str,f64,str,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str
"""count""",6.590154e6,"""6590154""","""6590154""",5.978354e6,6.590154e6,5.978354e6,"""5978354""",6.590154e6,6.590154e6,6.590154e6,6.590154e6,6.590154e6,6.590154e6,6.590154e6,6.590154e6,6.590154e6,6.590154e6,5.978354e6,5.978354e6,"""6590154""","""6590154""","""6590154""","""6590154""","""6590154""","""6590154"""
"""null_count""",0.0,"""0""","""0""",611800.0,0.0,611800.0,"""611800""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,611800.0,611800.0,"""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",1.756811,"""2024-03-02 23:29:08.306262""","""2024-03-02 23:45:29.766303""",1.332109,4.217783,2.189424,null,164.836847,163.769507,1.103447,18.39262,1.400976,0.48309,3.241934,0.526057,0.974779,26.894141,2.259032,0.136208,null,null,null,null,null,null
"""std""",0.430632,null,null,0.836458,281.631336,10.395004,null,64.242165,69.446392,0.622692,18.359656,1.804078,0.119625,3.941163,2.122793,0.221227,22.897657,0.826409,0.48108,null,null,null,null,null,null
"""min""",1.0,"""2002-12-31 22:17:10""","""2002-12-31 22:42:24""",0.0,0.0,1.0,"""N""",1.0,1.0,0.0,-999.0,-7.5,-0.5,-300.0,-84.3,-1.0,-1000.0,-2.5,-1.75,"""Bronx""","""Allerton/Pelham Gardens""","""Airports""","""Bronx""","""Allerton/Pelham Gardens""","""Airports"""
"""25%""",2.0,"""2024-02-16 23:06:24""","""2024-02-16 23:22:02""",1.0,1.0,1.0,null,132.0,113.0,1.0,8.61,0.0,0.5,0.0,0.0,1.0,15.48,2.5,0.0,null,null,null,null,null,null
"""50%""",2.0,"""2024-03-03 14:05:02""","""2024-03-03 14:23:32""",1.0,1.71,1.0,null,162.0,162.0,1.0,13.5,1.0,0.5,2.6,0.0,1.0,20.43,2.5,0.0,null,null,null,null,null,null
"""75%""",2.0,"""2024-03-17 12:36:18""","""2024-03-17 12:54:35""",1.0,3.2,1.0,null,234.0,234.0,1.0,21.2,2.5,0.5,4.12,0.0,1.0,29.04,2.5,0.0,null,null,null,null,null,null
"""max""",6.0,"""2024-04-01 00:34:55""","""2024-04-02 18:08:46""",9.0,222478.29,99.0,"""Y""",265.0,265.0,4.0,9792.0,14.25,35.84,999.99,163.0,1.0,9792.0,2.5,1.75,"""Unknown""","""Yorkville West""","""Yellow Zone""","""Unknown""","""Yorkville West""","""Yellow Zone"""


In [7]:
rides_df_raw.select(pl.all().null_count()/ pl.len())

vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pu_borough,pu_zone,pu_service_zone,do_borough,do_zone,do_service_zone
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.092835,0.0,0.092835,0.092835,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.092835,0.092835,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
cells_with_null = ["passenger_count", "rate_code_id", "store_and_fwd_flag", "congestion_surcharge", "airport_fee"]
rides_df_raw.select(
    pl.all_horizontal(pl.col(cells_with_null).is_null
)

passenger_count
f64
0.092835


### Plots of the target distribution

In [13]:
import hvplot.polars

In [14]:
rides_df_raw.hvplot.hist("tip_amount", bins=5000, xlim=(-1, 50))

:Histogram   [tip_amount]   (Count)